# Motor de qualidade orientado a contratos (DAMA-DMBOK)

Avalia as dimensões declaradas em cada contrato YAML sobre a camada **silver** e grava os resultados em `case_ifood.governanca.dq_resultados`:

| Dimensão | Origem no contrato | Regra |
| --- | --- | --- |
| Completude | `regras_qualidade.completude` | % de nulos por coluna ≤ `max_pct_nulo` |
| Validade | `regras_qualidade.validade` | nº de violações de cada expressão = 0 |
| Unicidade | `regras_qualidade.unicidade` | chave `trip_id` sem duplicatas |
| Consistência | `regras_qualidade.consistencia` | nº de violações = 0 |
| Pontualidade | `regras_qualidade.pontualidade` | linhas fora da janela temporal = 0 |

**Falha crítica interrompe o job** (`raise`). Nenhuma lógica específica de dataset: tudo vem do contrato.

**Investigação de falhas:** toda regra `FAIL` grava na coluna `exemplos` de `dq_resultados` um JSON com até 5 registros que violaram a regra — a chave de negócio (chave de unicidade do contrato) e as colunas citadas na expressão — para identificar exatamente qual registro caiu na regra.

In [0]:
import sys
from datetime import datetime, timezone

from pathlib import Path

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
candidatos = [Path(notebook_path), Path("/Workspace") / notebook_path.lstrip("/")]
REPO = None
for candidato in candidatos:
    for raiz in (candidato, *candidato.parents):
        if (raiz / "00_config" / "motores.py").is_file():
            REPO = str(raiz)
            break
    if REPO:
        break
if REPO is None:
    raise FileNotFoundError(f"Raiz do projeto nao localizada a partir do notebook: {notebook_path}")
sys.path.insert(0, str(Path(REPO) / "00_config"))
from motores import criar_tabelas_governanca, listar_contratos

CATALOGO = "case_ifood"

# Widgets (execução standalone ou via job)
dbutils.widgets.text("catalogo", CATALOGO)
dbutils.widgets.text("datasets", "")  # vazio = todos os contratos

catalogo = dbutils.widgets.get("catalogo")
datasets_filtro = [d.strip() for d in dbutils.widgets.get("datasets").split(",") if d.strip()]

criar_tabelas_governanca(catalogo, spark)
CONTRATOS = [c for c in listar_contratos()
             if not datasets_filtro or c["dataset"] in datasets_filtro]
print(f"Contratos carregados: {[c['dataset'] for c in CONTRATOS]}")

In [0]:
import json
import re

from pyspark.sql import functions as F

AVALIADO_EM = datetime.now(timezone.utc)
LIMITE_EXEMPLOS = 5  # registros de exemplo coletados por regra violada
resultados = []  # (dataset, tabela, dimensao, regra, metrica, status, exemplos)


def _registrar(dataset, tabela, dimensao, regra, metrica, ok, exemplos=None):
    resultados.append((dataset, tabela, dimensao, regra, float(metrica),
                       "PASS" if ok else "FAIL", exemplos))


def _exemplos_violacoes(df_violacoes, colunas_relevantes, chave,
                        limite=LIMITE_EXEMPLOS):
    """JSON com ate `limite` registros que violaram a regra.

    Cada exemplo traz a chave de negocio (chave de unicidade do contrato) e
    as colunas citadas na regra, para investigar o registro diretamente.
    """
    cols = list(dict.fromkeys(c for c in [chave, *colunas_relevantes]
                              if c in df_violacoes.columns))
    amostra = df_violacoes.select(*cols).limit(limite).collect()
    if not amostra:
        return None
    return json.dumps(
        [{c: (str(r[c]) if r[c] is not None else None) for c in cols}
         for r in amostra],
        ensure_ascii=False,
    )


def _colunas_da_regra(expressao, df):
    """Colunas da tabela citadas na expressao da regra."""
    tokens = set(re.findall(r"[A-Za-z_][A-Za-z0-9_]*", expressao))
    return [c for c in df.columns if c in tokens]


for contrato in CONTRATOS:
    ds = contrato["dataset"]
    tabela_silver = contrato["destinos"]["silver"]
    df = spark.read.table(tabela_silver)
    total = df.count()
    regras = contrato.get("regras_qualidade", {})
    chave = regras.get("unicidade", {}).get("chave", "trip_id")

    # --- Completude -------------------------------------------------------
    comp = regras.get("completude", {})
    max_nulo = float(comp.get("max_pct_nulo", 1.0))
    excessoes = {c: float(v) for c, v in comp.get("excessoes", {}).items()}
    colunas_comp = list(comp.get("colunas", [])) + list(excessoes.keys())
    for coluna in colunas_comp:
        limite = excessoes.get(coluna, max_nulo)
        nulos_df = df.filter(F.col(coluna).isNull())
        pct_nulo = nulos_df.count() / total if total else 0.0
        ok = pct_nulo <= limite
        exemplos = (_exemplos_violacoes(nulos_df, [coluna], chave)
                    if not ok else None)
        _registrar(ds, tabela_silver, "completude",
                   f"pct_nulo({coluna}) <= {limite}", pct_nulo, ok, exemplos)

    # --- Validade ---------------------------------------------------------
    for regra in regras.get("validade", {}).get("regras", []):
        viol_df = df.filter(f"NOT ({regra['expressao']})")
        violacoes = viol_df.count()
        ok = violacoes == 0
        exemplos = (_exemplos_violacoes(
                        viol_df, _colunas_da_regra(regra["expressao"], df), chave)
                    if not ok else None)
        _registrar(ds, tabela_silver, "validade",
                   regra["expressao"], violacoes, ok, exemplos)

    # --- Unicidade --------------------------------------------------------
    dups_df = (df.groupBy(chave).agg(F.count("*").alias("ocorrencias"))
               .filter("ocorrencias > 1"))
    dups = total - df.select(chave).distinct().count()
    ok = dups == 0
    exemplos = (_exemplos_violacoes(dups_df, ["ocorrencias"], chave)
                if not ok else None)
    _registrar(ds, tabela_silver, "unicidade", f"chave unica({chave})", dups, ok, exemplos)

    # --- Consistência -----------------------------------------------------
    for i, expressao in enumerate(regras.get("consistencia", {}).get("regras", [])):
        viol_df = df.filter(f"NOT ({expressao})")
        violacoes = viol_df.count()
        ok = violacoes == 0
        exemplos = (_exemplos_violacoes(
                        viol_df, _colunas_da_regra(expressao, df), chave)
                    if not ok else None)
        _registrar(ds, tabela_silver, "consistencia", expressao, violacoes, ok, exemplos)

    # --- Pontualidade -----------------------------------------------------
    pont = regras.get("pontualidade", {})
    janela_inicio = pont.get("janela_inicio")
    janela_fim = pont.get("janela_fim")
    if janela_inicio or janela_fim:
        cond = []
        if janela_inicio:
            cond.append(f"pickup_datetime >= '{janela_inicio}'")
        if janela_fim:
            cond.append(f"pickup_datetime < '{janela_fim}'")
        fora_df = df.filter(" OR ".join(f"NOT ({c})" for c in cond))
        fora = fora_df.count()
        ok = fora == 0
        exemplos = (_exemplos_violacoes(
                        fora_df, _colunas_da_regra(" ".join(cond), df), chave)
                    if not ok else None)
        _registrar(ds, tabela_silver, "pontualidade",
                   f"pickup_datetime entre {janela_inicio} e {janela_fim}", fora, ok, exemplos)

df_res = spark.createDataFrame(
    [(ds, tb, dim, regra, met, st, AVALIADO_EM, exemplos)
     for ds, tb, dim, regra, met, st, exemplos in resultados],
    "dataset STRING, tabela STRING, dimensao STRING, regra STRING, metrica DOUBLE, "
    "status STRING, avaliado_em TIMESTAMP, exemplos STRING",
)
df_res.write.mode("append").saveAsTable(f"{catalogo}.governanca.dq_resultados")
df_res.orderBy("status", "dataset", "dimensao").show(100, truncate=False)

falhas = [r for r in resultados if r[5] == "FAIL"]
if falhas:
    resumo = [(r[0], r[2], r[3], r[4]) for r in falhas]
    raise Exception(
        f"QUALIDADE FALHOU: {len(falhas)} regras violadas: {resumo}. "
        f"Registros de exemplo em {catalogo}.governanca.dq_resultados (coluna exemplos)."
    )
print(f"✅ Qualidade OK: {len(resultados)} regras avaliadas, nenhuma violação crítica.")